# 01 — Train all three tiers

Trains Tier 1 (LightGBM), Tier 2 (frozen-encoder cross-attention fusion) and
Tier 3 (Tier 1 + pocket-occupancy features) across all four splits, and reports
PR-AUC, MCC and Brier score with error bars over 5 seeds.

## Why error bars, and why over *split* seeds

A single split of ~250 test rows carries a standard error on PR-AUC of several
points. Two methods differing by 0.02 on one seed are indistinguishable. We
therefore vary the **split seed and the model seed together**, because at this
sample size split variance dominates — reporting only model-seed variance
understates the uncertainty badly and makes tiers look more separable than a
rerun would show.

## Why a ceiling column

The synthetic cohort exposes `oracle_p`, the true generative probability. Its
PR-AUC is the **Bayes ceiling**: the best any model could achieve against this
much label noise. Without it, a PR-AUC of 0.45 is uninterpretable — it could be
a weak model or a noisy problem. `fraction_of_ceiling` is the honest summary.


> ## ⚠ These results are computed on SYNTHETIC data
>
> The curated benchmark this package is designed to consume
> ([`amenabench`](https://github.com/xX-its-amit-Xx/amenabench)) currently
> contains only a `LICENSE` file — there is no data in it. Every number and
> figure below is therefore produced by
> `foldrescue.data.SyntheticCohortAdapter`, a seeded generator whose labels
> encode a mixture **we chose**.
>
> **What this notebook demonstrates:** that the pipeline runs end to end, and
> that the diagnostics behave as designed against a known ground truth.
>
> **What it does not demonstrate:** anything about pharmacological chaperone
> biology. These are not findings. Do not cite these numbers.


In [1]:
import sys, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")          # cookbook/ -> repo root
sys.path.insert(0, ".")
from _viz import (SERIES, TIER_COLOR, BLOCK_COLOR, INK, INK_SECONDARY,
                  INK_MUTED, use_style, save, label_bars)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
use_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("numpy", np.__version__, "| pandas", pd.__version__)


numpy 2.5.3 | pandas 3.0.5


In [2]:
from foldrescue.data import load_cohort
from foldrescue.features import FeatureSpec, build_features
from foldrescue.models import build_model
from foldrescue.evaluate import evaluate_predictions, aggregate_over_seeds
from foldrescue.embeddings import build_embedding_matrix

SEEDS   = [0, 1, 2, 3, 4]
SPLITS  = ["random", "gene_holdout", "compound_holdout", "scaffold"]
ECFP    = 256
CACHE   = "../.foldrescue_cache/embeddings"

# Tier 3 is the only tier given the structural block -- that is what defines it.
BLOCKS = {
    "tier1_gbm":       ("stability", "ligand", "interaction"),
    "tier3_structure": ("stability", "structural", "ligand", "interaction"),
}
print("tiers:", list(BLOCKS) + ["tier2_fusion"])
print("splits:", SPLITS)
print("seeds:", SEEDS)

tiers: ['tier1_gbm', 'tier3_structure', 'tier2_fusion']
splits: ['random', 'gene_holdout', 'compound_holdout', 'scaffold']
seeds: [0, 1, 2, 3, 4]


### Is Tier 2 running on real ESM-2 / ChemBERTa?

Check this before reading any Tier 2 number. This environment's proxy allowlist
does not include `huggingface.co` (403 on CONNECT), so pretrained weights cannot
be fetched and the **surrogate** backend is used: hashed k-mer counts for the
protein window, a seeded random projection of Morgan bits for the compound.

That is a deterministic stand-in, **not** a language-model representation. Tier 2
numbers below therefore exercise the *fusion architecture* and say nothing
whatsoever about ESM-2 or ChemBERTa. Where `huggingface.co` is reachable, pass
`protein_backend="esm2", compound_backend="chemberta"` to use the real encoders.

In [3]:
_X, emb_prov = build_embedding_matrix(load_cohort("synthetic", seed=0), cache_root=CACHE)
for k, v in emb_prov.items():
    print(f"{k:22s} {v}")
assert emb_prov["surrogate"], "expected the surrogate backend in this environment"
print("\nTier 2 embedding width:", _X.shape[1])

2026-10-10 11:54:59,062 INFO    foldrescue.data: synthetic cohort: 176 variants x 12 compounds -> 1308 pairs, prevalence 0.312


2026-10-10 11:54:59,071 INFO    foldrescue.embeddings: resumed embedding cache ../.foldrescue_cache/embeddings/protein_02cad52c26b0847c with 880 entities


2026-10-10 11:54:59,074 INFO    foldrescue.embeddings: all 176 entities already cached (../.foldrescue_cache/embeddings/protein_02cad52c26b0847c)


2026-10-10 11:54:59,077 INFO    foldrescue.embeddings: resumed embedding cache ../.foldrescue_cache/embeddings/compound_576d99d7e9a7ee2f with 12 entities


2026-10-10 11:54:59,077 INFO    foldrescue.embeddings: all 12 entities already cached (../.foldrescue_cache/embeddings/compound_576d99d7e9a7ee2f)


2026-10-10 11:54:59,085 WARNING foldrescue.embeddings: Tier 2 is running on SURROGATE embeddings (hashed-kmer-surrogate / morgan-projection-surrogate). These are deterministic hashes, NOT pretrained language model representations. Results exercise the fusion architecture and say nothing about ESM-2 or ChemBERTa.


protein_backend        hashed
protein_model          hashed-kmer-surrogate
compound_backend       hashed
compound_model         morgan-projection-surrogate
surrogate              True
unfreeze_encoders      False

Tier 2 embedding width: 704


In [4]:
rows = []
t_start = time.time()

for seed in SEEDS:
    cohort = load_cohort("synthetic", seed=seed)
    y_all = cohort.pairs["label"].to_numpy()

    # Tree tiers
    feats = {t: build_features(cohort, spec=FeatureSpec(blocks=b, ecfp_bits=ECFP))
             for t, b in BLOCKS.items()}
    # Fusion tier embeddings (cached + resumable across seeds)
    Xe, _ = build_embedding_matrix(cohort, cache_root=CACHE)

    for split in SPLITS:
        s = cohort.split(split)
        for tier in list(BLOCKS) + ["tier2_fusion"]:
            if tier == "tier2_fusion":
                X = Xe
                model = build_model(tier, protein_dim=320, compound_dim=384,
                                    epochs=80, random_state=seed)
                model.fit(X[s["train"]], y_all[s["train"]],
                          eval_set=(X[s["calib"]], y_all[s["calib"]]))
                names = None
            else:
                fm = feats[tier]
                X, names = fm.X, fm.feature_names
                model = build_model(tier, random_state=seed)
                model.fit(X[s["train"]], y_all[s["train"]],
                          eval_set=(X[s["calib"]], y_all[s["calib"]]),
                          feature_names=names)

            m = evaluate_predictions(
                y_all[s["test"]], model.predict_proba(X[s["test"]]),
                split=split, tier=tier,
                y_calib=y_all[s["calib"]],
                p_calib=model.predict_proba(X[s["calib"]]),
                oracle_p_test=cohort.oracle_p[s["test"]],
            )
            row = m.as_row(); row["seed"] = seed
            rows.append(row)

raw = pd.DataFrame(rows)
print(f"{len(raw)} fits in {time.time()-t_start:.1f}s")
raw.head()

2026-10-10 11:54:59,404 INFO    foldrescue.data: synthetic cohort: 176 variants x 12 compounds -> 1308 pairs, prevalence 0.312


2026-10-10 11:54:59,426 INFO    foldrescue.features: built features: (1308, 289) (float64)


2026-10-10 11:54:59,448 INFO    foldrescue.features: built features: (1308, 300) (float64)


2026-10-10 11:54:59,458 INFO    foldrescue.embeddings: resumed embedding cache ../.foldrescue_cache/embeddings/protein_02cad52c26b0847c with 880 entities


2026-10-10 11:54:59,460 INFO    foldrescue.embeddings: all 176 entities already cached (../.foldrescue_cache/embeddings/protein_02cad52c26b0847c)


...

2026-10-10 11:55:57,929 INFO    foldrescue.models: tier2: early stop at epoch 20 (best val 0.9638)


2026-10-10 11:56:00,607 INFO    foldrescue.models: tier2: early stop at epoch 21 (best val 0.8266)


2026-10-10 11:56:03,633 INFO    foldrescue.models: tier2: early stop at epoch 23 (best val 0.9580)


60 fits in 64.6s


,split,tier,n_test,prevalence,pr_auc,roc_auc,mcc,brier,ece,threshold,n_predicted_positive,oracle_pr_auc,fraction_of_ceiling,seed
0,random,tier1_gbm,262,0.328244,0.438505,0.625727,0.178755,0.214906,0.065334,0.305133,116,0.667534,0.324974,0
1,random,tier3_structure,262,0.328244,0.544475,0.670719,0.189515,0.201796,0.064168,0.291877,129,0.667534,0.637305,0
2,random,tier2_fusion,262,0.328244,0.375632,0.555299,0.013437,0.262421,0.206428,0.468165,202,0.667534,0.139669,0
3,gene_holdout,tier1_gbm,373,0.294906,0.465565,0.714345,0.249923,0.192034,0.083817,0.374463,88,0.648185,0.483072,0
4,gene_holdout,tier3_structure,373,0.294906,0.465412,0.687522,0.197351,0.197480,0.109544,0.369842,47,0.648185,0.482638,0


### Results

`pr_auc` is always read next to `prevalence`: PR-AUC is only interpretable
relative to the base rate, since the no-skill baseline for average precision
*is* the prevalence.

In [5]:
agg = aggregate_over_seeds(raw.to_dict("records"), by=("split", "tier"))
view = agg[["split", "tier", "prevalence_mean", "pr_auc_mean", "pr_auc_std",
            "mcc_mean", "mcc_std", "brier_mean", "brier_std",
            "oracle_pr_auc_mean", "fraction_of_ceiling_mean"]].copy()
view.columns = ["split", "tier", "prev", "PR-AUC", "+/-", "MCC", "+/-",
                "Brier", "+/- ", "ceiling", "frac_ceiling"]
view = view.round(4)
print(view.to_string(index=False))
view.to_csv("../results/DEMO_synthetic_tier_comparison.csv", index=False)

           split            tier   prev  PR-AUC    +/-    MCC    +/-  Brier   +/-   ceiling  frac_ceiling
compound_holdout       tier1_gbm 0.3176  0.4428 0.0424 0.1668 0.0607 0.2131 0.0225   0.6104        0.4221
compound_holdout    tier2_fusion 0.3176  0.3236 0.0480 0.0401 0.0682 0.2446 0.0048   0.6104        0.0218
compound_holdout tier3_structure 0.3176  0.4267 0.0697 0.1231 0.0438 0.2169 0.0184   0.6104        0.3941
    gene_holdout       tier1_gbm 0.3123  0.4347 0.0418 0.1697 0.1030 0.2048 0.0137   0.5982        0.4075
    gene_holdout    tier2_fusion 0.3123  0.3337 0.0263 0.0492 0.0605 0.2523 0.0111   0.5982        0.0672
    gene_holdout tier3_structure 0.3123  0.4265 0.0475 0.1785 0.0521 0.2101 0.0129   0.5982        0.3881
          random       tier1_gbm 0.2976  0.4070 0.0778 0.1427 0.0657 0.2016 0.0195   0.5908        0.3817
          random    tier2_fusion 0.2976  0.3608 0.0595 0.0984 0.0617 0.2449 0.0120   0.5908        0.2175
          random tier3_structure 0.2976  0.436

### PR-AUC by split and tier

Grey marks the no-skill baseline (the prevalence) and the dashed rule marks the
Bayes ceiling. The distance between those two is all the signal that exists;
anything a bar does outside that band is noise.

In [6]:
fig, axes = plt.subplots(1, len(SPLITS), figsize=(14.5, 3.9), sharey=True)
tiers = ["tier1_gbm", "tier2_fusion", "tier3_structure"]

for ax, split in zip(axes, SPLITS, strict=True):
    sub = agg[agg["split"] == split].set_index("tier")
    xs = np.arange(len(tiers))
    means = [sub.loc[t, "pr_auc_mean"] for t in tiers]
    errs  = [sub.loc[t, "pr_auc_std"] for t in tiers]
    bars = ax.bar(xs, means, yerr=errs, capsize=3.5, width=0.62,
                  color=[TIER_COLOR[t] for t in tiers], zorder=3)
    for b in bars:
        b.set_linewidth(0)
    prev = sub["prevalence_mean"].iloc[0]
    ceil = sub["oracle_pr_auc_mean"].iloc[0]
    ax.axhline(prev, color=INK_MUTED, lw=1.6, zorder=2)
    ax.axhline(ceil, color=INK, lw=1.3, ls=(0, (4, 3)), zorder=2)
    label_bars(ax, bars, means, dy=max(errs) + 0.012)
    ax.set_title(split.replace("_", " "))
    ax.set_xticks(xs)
    ax.set_xticklabels(["T1\ngbm", "T2\nfusion", "T3\nstruct"])
    ax.set_ylim(0, 0.92)

axes[0].set_ylabel("PR-AUC (mean +/- SD over 5 seeds)")

# The two reference lines need naming in the figure itself: an annotation
# placed past the last bar gets clipped by the axes, which left both rules
# unexplained in an earlier render.
from matplotlib.lines import Line2D
fig.legend(
    handles=[
        Line2D([], [], color=INK, lw=1.3, ls=(0, (4, 3)),
               label="Bayes ceiling (oracle PR-AUC)"),
        Line2D([], [], color=INK_MUTED, lw=1.6,
               label="no-skill baseline (= prevalence)"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, -0.17), ncol=2, frameon=False,
)
fig.suptitle("PR-AUC by split and model tier", y=1.04, fontsize=12, color=INK)
save(fig, "01_pr_auc_by_split_tier.png")
plt.show()

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


wrote results/figures/01_pr_auc_by_split_tier.png


### Does the fancy tier win?

In [7]:
piv = agg.pivot(index="split", columns="tier", values="pr_auc_mean").round(4)
sd  = agg.pivot(index="split", columns="tier", values="pr_auc_std").round(4)
print("PR-AUC means:\n", piv.to_string(), "\n")
print("PR-AUC SDs:\n", sd.to_string(), "\n")

for split in SPLITS:
    best = piv.loc[split].idxmax()
    t1, t3 = piv.loc[split, "tier1_gbm"], piv.loc[split, "tier3_structure"]
    pooled = float(np.hypot(sd.loc[split, "tier1_gbm"], sd.loc[split, "tier3_structure"]))
    sep = "separable" if abs(t3 - t1) > pooled else "NOT separable given seed variance"
    print(f"{split:18s} best={best:16s} tier3-tier1={t3-t1:+.4f}  ({sep})")

PR-AUC means:
 tier              tier1_gbm  tier2_fusion  tier3_structure
split                                                     
compound_holdout     0.4428        0.3236           0.4267
gene_holdout         0.4347        0.3337           0.4265
random               0.4070        0.3608           0.4365
scaffold             0.4226        0.3279           0.4175 

PR-AUC SDs:
 tier              tier1_gbm  tier2_fusion  tier3_structure
split                                                     
compound_holdout     0.0424        0.0480           0.0697
gene_holdout         0.0418        0.0263           0.0475
random               0.0778        0.0595           0.0818
scaffold             0.0579        0.0537           0.0654 

random             best=tier3_structure  tier3-tier1=+0.0295  (NOT separable given seed variance)
gene_holdout       best=tier1_gbm        tier3-tier1=-0.0082  (NOT separable given seed variance)
compound_holdout   best=tier1_gbm        tier3-tier1=-0.0161  (N

### What to take from this

Read the printed comparison above rather than a remembered headline. Three
things are worth stating regardless of the exact numbers:

1. **The interpretable baseline is hard to beat at this sample size.** With
   ~10³ pairs, gradient-boosted trees over hand-built features are a strong
   model, and a cross-attention fusion head on frozen (here, surrogate)
   embeddings has far more capacity than the data supports. This is the
   expected result, not a disappointing one — it is why Tier 1 exists.
2. **Differences smaller than the pooled seed SD are not differences.** The
   separability check above says so explicitly for each split.
3. **The split choice matters more than the tier choice.** `random` is the most
   optimistic number in the table and answers the least useful question;
   `gene_holdout` answers "a protein I have never trained on", which is what a
   prospective query actually is.
